# Image Extract
### Purpose: unzip archives (or use already-extracted images), strip `__MACOSX`, build train masks from annotations.


In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
_start = Path(os.getenv("PROJECT_ROOT") or Path.cwd()).resolve()
root = next(p for p in [_start, *_start.parents] if (p / "config.yaml").exists())
sys.path[:0] = [str(root), str(root / "src")]

import random
import shutil
import zipfile

from src.data.annotations import load_json_annotations
from src.data.masks import save_mask
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t

config = Config.load(root=root)
init_notebook(config.train.seed)

train_zip = config.paths.train_images_zip
eval_zip = config.paths.eval_images_zip
train_dir = config.paths.train_images
eval_dir = config.paths.eval_images
mask_dir = config.paths.train_masks



=== init_notebook ===
Done


#### Extract images and remove `__MACOSX` folders


In [2]:
def _image_count(folder: Path) -> int:
    if not folder or not Path(folder).exists():
        return 0
    folder = Path(folder)
    return sum(1 for _ in folder.glob("*.tif")) + sum(1 for _ in folder.glob("*.tiff")) + sum(
        1 for _ in folder.glob("*.png")
    )


# Mapping of zip files to their extraction targets
extraction_map = [
    ("train", train_zip, train_dir),
    ("eval", eval_zip, eval_dir),
]

for label, zip_path, extract_to in extraction_map:
    extract_to = Path(extract_to) if extract_to else None
    zip_path = Path(zip_path) if zip_path else None
    n_imgs = _image_count(extract_to) if extract_to else 0

    if zip_path and zip_path.exists():
        p("Working", str(zip_path))
        extract_to.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_to)
            src_rel = zip_path.resolve().relative_to(root)
            dst_rel = extract_to.resolve().relative_to(root)
            p("Extracted", f"{src_rel} -> {dst_rel}")

        for m in extract_to.rglob("__MACOSX"):
            if m.is_dir():
                shutil.rmtree(m)
                p("Removed", str(m.resolve().relative_to(root)))

        p(f"✓ {label} images", f"{_image_count(extract_to)} files in {extract_to.name}")
        p()
    elif n_imgs > 0:
        p(f"✓ {label} zip skipped", "not found (OK)")
        p(f"✓ {label} images found", f"{n_imgs} files in {extract_to}")
        p()
    else:
        p(f"✗ {label} missing", f"no zip and no images at {extract_to}")
        p()


Working: /Volumes/Colibri/Projects/cap6415-tree-canopy-detection/data/public_sample/raw/data.zip
Extracted: data/public_sample/raw/data.zip -> data/public_sample/train_images
Removed: data/public_sample/train_images/__MACOSX
✓ train images: 14 files in train_images

✓ eval zip skipped: not found (OK)
✓ eval images found: 4 files in /Volumes/Colibri/Projects/cap6415-tree-canopy-detection/data/public_sample/evaluation_images



#### Build masks from annotations


In [3]:
annotations_path = config.paths.annotations
entries = load_json_annotations(annotations_path)
mask_dir.mkdir(parents=True, exist_ok=True)

p("Annotations", f"{len(entries)} images")
p("Mask dir", str(mask_dir))

saved = 0
skipped = 0
for entry in entries:
    image_path = train_dir / entry.image_path.name
    if not image_path.exists():
        skipped += 1
        continue

    mask = entry.to_mask()
    save_path = mask_dir / entry.image_path.name
    save_mask(mask, save_path)
    saved += 1

p("✓ Masks saved", f"{saved}")
if skipped:
    p("⚠ Skipped", f"{skipped} (image file missing)")
else:
    p("✓ Skipped", "0 (all images present)")
p("Done", "build masks from annotations")


Annotations: 14 images
Mask dir: /Volumes/Colibri/Projects/cap6415-tree-canopy-detection/data/public_sample/train_masks
✓ Masks saved: 14
✓ Skipped: 0 (all images present)
Done: build masks from annotations
